In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder  
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler  
from joblib import Parallel, delayed  
import joblib
import os
import gc  
import time 

In [2]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [3]:
path = 'reddit_depression_dataset.csv'
dataset = pd.read_csv(path)

features_no = dataset.shape[1]
print("The number of features in this dataset is: ", features_no)

data_points = len(dataset)
print(f"The number of data points in this dataset is: ", data_points)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_12996\2284969625.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv(path)


The number of features in this dataset is:  8
The number of data points in this dataset is:  2470778


In [4]:
dataset.head()

,Unnamed: 0,subreddit,title,body,upvotes,created_utc,num_comments,label
0,47951,DeepThoughts,Deep thoughts underdog,"Only when we start considering ourselves, the ...",4.0,1.405309e+09,NaN,0.0
1,47952,DeepThoughts,"I like this sub, there's only two posts yet I ...",Anyway: Human Morality is a joke so long as th...,4.0,1.410568e+09,1.0,0.0
2,47957,DeepThoughts,Rebirth!,Hello. \nI am the new guy in charge here (Besi...,6.0,1.416458e+09,1.0,0.0
3,47959,DeepThoughts,"""I want to be like water. I want to slip throu...",NaN,25.0,1.416512e+09,2.0,0.0
4,47960,DeepThoughts,Who am I?,You could take any one cell in my body and kil...,5.0,1.416516e+09,4.0,0.0


In [5]:
dataset.shape

(2470778, 8)

In [6]:
dataset.describe()

,upvotes,created_utc,num_comments,label
count,2.470714e+06,2.470672e+06,2.356801e+06,2.470672e+06
mean,6.260360e+01,1.566474e+09,1.526266e+01,1.944455e-01
std,9.474559e+02,6.987755e+07,7.867238e+01,3.957733e-01
min,0.000000e+00,1.201314e+09,-3.000000e+00,0.000000e+00
25%,5.000000e+00,1.552205e+09,4.000000e+00,0.000000e+00
50%,7.000000e+00,1.578371e+09,7.000000e+00,0.000000e+00
75%,1.000000e+01,1.611540e+09,1.400000e+01,0.000000e+00
max,1.288660e+05,1.672531e+09,2.113100e+04,1.000000e+00


In [7]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2470778 entries, 0 to 2470777
Data columns (total 8 columns):
 #   Column        Dtype  
---  ------        -----  
 0   Unnamed: 0    object 
 1   subreddit     object 
 2   title         object 
 3   body          object 
 4   upvotes       float64
 5   created_utc   float64
 6   num_comments  float64
 7   label         float64
dtypes: float64(4), object(4)
memory usage: 150.8+ MB


In [8]:
dataset.isnull().sum()

Unnamed: 0           3
subreddit           20
title               23
body            461051
upvotes             64
created_utc        106
num_comments    113977
label              106
dtype: int64

In [9]:
dataset.duplicated().any()

np.False_

MISSING VALUES HANDLING

In [10]:
dataset = dataset.drop(columns=['Unnamed: 0'], errors='ignore')
dataset = dataset.dropna(subset=['label'])
dataset['body'] = dataset['body'].fillna(dataset['title']).fillna('')
dataset['subreddit'] = dataset['subreddit'].fillna('unknown')
dataset['title'] = dataset['title'].fillna('')
dataset['num_comments'] = dataset['num_comments'].fillna(0)
dataset['upvotes'] = dataset['upvotes'].fillna(0)
dataset['created_utc'] = dataset['created_utc'].fillna(dataset['created_utc'].median())

print("Missing values after handling:\n", dataset.isnull().sum())

Missing values after handling:
 subreddit       0
title           0
body            0
upvotes         0
created_utc     0
num_comments    0
label           0
dtype: int64


TEXT PREPROCESSING

In [11]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    """Clean text by lowercasing, removing URLs/punctuation, tokenizing, removing stopwords, and lemmatizing."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) 
    text = re.sub(r'[^\\w\\s]', '', text) 
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

dataset['cleaned_title'] = dataset['title'].apply(clean_text)
dataset['cleaned_body'] = dataset['body'].apply(clean_text)

dataset['combined_text'] = dataset['cleaned_title'] + ' ' + dataset['cleaned_body']

In [12]:
dataset.to_csv('processed_data.csv', index=False)

In [13]:
from textblob import TextBlob

def get_sentiment(text):
    """Calculate sentiment polarity using TextBlob."""
    return TextBlob(text).sentiment.polarity

dataset['sentiment'] = dataset['combined_text'].apply(get_sentiment)

dataset.to_csv('data_with_sentiment.csv', index=False)

In [14]:
dataset['text_length'] = dataset['combined_text'].apply(len)
dataset['word_count'] = dataset['combined_text'].apply(lambda x: len(x.split()))

dataset.to_csv('data_with_text_features.csv', index=False)

In [15]:
dataset['created_date'] = pd.to_datetime(dataset['created_utc'], unit='s')
dataset['year'] = dataset['created_date'].dt.year
dataset['month'] = dataset['created_date'].dt.month
dataset['day_of_week'] = dataset['created_date'].dt.dayofweek

dataset.to_csv('data_with_temporal_features.csv', index=False)

In [16]:
import numpy as np

dataset['log_upvotes'] = np.log1p(dataset['upvotes'])
dataset['log_num_comments'] = np.log1p(dataset['num_comments'])

dataset['upvotes_per_comment'] = dataset['upvotes'] / (dataset['num_comments'] + 1)

dataset.to_csv('data_with_engagement_features.csv', index=False)

C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [17]:
le = LabelEncoder()
dataset['subreddit_encoded'] = le.fit_transform(dataset['subreddit'])

dataset.to_csv('data_with_encoded_subreddit.csv', index=False)

In [18]:
dataset['upvotes'] = np.clip(dataset['upvotes'], 0, dataset['upvotes'].quantile(0.99))
dataset['num_comments'] = np.clip(dataset['num_comments'], 0, dataset['num_comments'].quantile(0.99))

dataset.to_csv('data_with_capped_values.csv', index=False)

In [19]:
dataset['label'] = dataset['label'].astype('int32')
dataset['upvotes'] = dataset['upvotes'].astype('int32')
dataset['num_comments'] = dataset['num_comments'].astype('int32')
dataset['year'] = dataset['year'].astype('int32')
dataset['month'] = dataset['month'].astype('int32')
dataset['day_of_week'] = dataset['day_of_week'].astype('int32')

dataset.to_csv('data_with_optimized_types.csv', index=False)

TEXT VECTORIZAZTION

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_text = tfidf.fit_transform(dataset['combined_text'])

from scipy.sparse import save_npz
save_npz('data_with_tfidf_features.npz', X_text)

In [21]:
from scipy.sparse import hstack

numerical_features = ['sentiment', 'text_length', 'word_count', 'log_upvotes',
                      'log_num_comments', 'upvotes_per_comment', 'subreddit_encoded',
                      'year', 'month', 'day_of_week']
X_numerical = dataset[numerical_features]

from scipy.sparse import csr_matrix
X_numerical_sparse = csr_matrix(X_numerical.values)


X = hstack([X_text, X_numerical_sparse])
y = dataset['label']


save_npz('final_feature_matrix.npz', X)
y.to_csv('final_labels.csv', index=False)

In [22]:
print("Class distribution:\n", y.value_counts())


plt.figure(figsize=(6, 4))
sns.countplot(x=y)
plt.title('Class Distribution of Labels')
plt.xlabel('Label')
plt.ylabel('Count')
plt.savefig('class_distribution.png')
plt.close()

Class distribution:
 label
0    1990261
1     480411
Name: count, dtype: int64


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)

save_npz('X_train.npz', X_train)
save_npz('X_test.npz', X_test)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

Training set shape: (1976537, 5010) (1976537,)
Test set shape: (494135, 5010) (494135,)


SHORTEN THE DATASET

In [24]:
import pandas as pd
from sklearn.utils import resample

dataset = pd.read_csv('F:/427_Project/data_with_encoded_subreddit.csv')

depressed = dataset[dataset['label'] == 1]
non_depressed = dataset[dataset['label'] == 0]

n_samples = 25000  
depressed_sub = resample(depressed, replace=False, n_samples=n_samples, random_state=42)
non_depressed_sub = resample(non_depressed, replace=False, n_samples=n_samples, random_state=42)

subset = pd.concat([depressed_sub, non_depressed_sub]).reset_index(drop=True)

subset.to_csv('F:/427_Project/subset_data.csv', index=False)
print("Subset shape:", subset.shape)

Subset shape: (50000, 21)


In [25]:
from scipy.sparse import hstack, save_npz

dataset = pd.read_csv('F:/427_Project/subset_data.csv')

dataset['body'] = dataset['body'].fillna(dataset['title'])
dataset['created_utc'] = dataset['created_utc'].fillna(dataset['created_utc'].median())
dataset['num_comments'] = dataset['num_comments'].fillna(0)
dataset['upvotes'] = dataset['upvotes'].fillna(0)

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text) 
    text = re.sub(r'[^\w\s]', '', text) 
    text = re.sub(r'\s+', ' ', text).strip()  
    return text

dataset['cleaned_text'] = dataset['body'].apply(clean_text)

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_text = tfidf.fit_transform(dataset['cleaned_text'])

dataset['upvotes'] = dataset['upvotes'].clip(lower=0, upper=dataset['upvotes'].quantile(0.95))
dataset['num_comments'] = dataset['num_comments'].clip(lower=0, upper=dataset['num_comments'].quantile(0.95))


dataset['log_upvotes'] = np.log1p(dataset['upvotes'])
dataset['log_num_comments'] = np.log1p(dataset['num_comments'])


dataset['upvotes_per_comment'] = np.where(dataset['num_comments'] == 0, 0, dataset['upvotes'] / dataset['num_comments'])
dataset['upvotes_per_comment'] = dataset['upvotes_per_comment'].clip(upper=dataset['upvotes_per_comment'].quantile(0.95))

dataset['sentiment'] = dataset['cleaned_text'].apply(lambda x: TextBlob(x).sentiment.polarity if x else 0)


dataset['text_length'] = dataset['cleaned_text'].apply(lambda x: len(x.split()))


numerical_features = dataset[['log_upvotes', 'log_num_comments', 'upvotes_per_comment', 'sentiment', 'text_length']].values
X_numerical = np.nan_to_num(numerical_features, nan=0.0, posinf=1e6, neginf=-1e6)  # Handle NaN and inf
X = hstack([X_text, X_numerical])

save_npz('F:/427_Project/subset_final_feature_matrix.npz', X)
dataset[['label']].to_csv('F:/427_Project/subset_final_labels.csv', index=False)
print("Preprocessed subset shape:", X.shape)

Preprocessed subset shape: (50000, 5005)


In [26]:
from scipy.sparse import load_npz, issparse


X = load_npz('F:/427_Project/subset_final_feature_matrix.npz')
y = pd.read_csv('F:/427_Project/subset_final_labels.csv')['label'].values

RANDOM FOREST

In [27]:
from sklearn.ensemble import RandomForestClassifier

X = load_npz('F:/427_Project/subset_final_feature_matrix.npz')
y = pd.read_csv('F:/427_Project/subset_final_labels.csv')['label'].values


if issparse(X):
    print("NaN values in X:", np.isnan(X.data).any())
    print("Infinite values in X:", np.isinf(X.data).any())
    if np.isnan(X.data).any():
        print("Replacing NaN with 0.")
        X.data = np.nan_to_num(X.data, nan=0.0)
    if np.isinf(X.data).any():
        print("Replacing infinite values with 1e6.")
        X.data = np.where(np.isinf(X.data), 1e6, X.data)
    X = X.astype(np.float32)
else:
    print("NaN values in X:", np.isnan(X).any())
    print("Infinite values in X:", np.isinf(X).any())
    if np.isnan(X).any():
        print("Replacing NaN with 0.")
        X = np.nan_to_num(X, nan=0.0)
    if np.isinf(X).any():
        print("Replacing infinite values with 1e6.")
        X = np.where(np.isinf(X), 1e6, X)
    X = X.astype(np.float32)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

NaN values in X: False
Infinite values in X: False
Classification Report:
               precision    recall  f1-score   support

         0.0       0.79      0.88      0.83      5000
         1.0       0.86      0.76      0.81      5000

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000

ROC-AUC Score: 0.90418438
Confusion Matrix:
 [[4380  620]
 [1198 3802]]


LOGISTIC REGRESSION

In [28]:
from scipy.sparse import load_npz
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

X = load_npz('F:/427_Project/subset_final_feature_matrix.npz')
y = pd.read_csv('F:/427_Project/subset_final_labels.csv')['label']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)

model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1] 

print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Non-Depressed', 'Depressed']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Depressed', 'Depressed'], 
            yticklabels=['Non-Depressed', 'Depressed'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('confusion_matrix.png')
plt.close()

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--') 
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='best')
plt.savefig('roc_curve.png')
plt.close()

import joblib
joblib.dump(model, 'logistic_regression_model.pkl')
print("Model saved as 'logistic_regression_model.pkl'")

Training set shape: (40000, 5005) (40000,)
Test set shape: (10000, 5005) (10000,)


C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classification Report:

               precision    recall  f1-score   support

Non-Depressed       0.88      0.91      0.89      5000
    Depressed       0.91      0.87      0.89      5000

     accuracy                           0.89     10000
    macro avg       0.89      0.89      0.89     10000
 weighted avg       0.89      0.89      0.89     10000

Model saved as 'logistic_regression_model.pkl'


KNN

In [29]:

from scipy.sparse import load_npz
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score


X = load_npz('F:/427_Project/subset_final_feature_matrix.npz')
y = pd.read_csv('F:/427_Project/subset_final_labels.csv')['label']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)

knn = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
knn.fit(X_train, y_train)


y_pred = knn.predict(X_test)
y_pred_proba = knn.predict_proba(X_test)[:, 1]  

print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Non-Depressed', 'Depressed']))


cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Depressed', 'Depressed'], 
            yticklabels=['Non-Depressed', 'Depressed'])
plt.title('KNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('knn_confusion_matrix.png')
plt.close()

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')  
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('KNN ROC Curve')
plt.legend(loc='best')
plt.savefig('knn_roc_curve.png')
plt.close()

import joblib
joblib.dump(knn, 'knn_model.pkl')

Training set shape: (40000, 5005) (40000,)
Test set shape: (10000, 5005) (10000,)
Classification Report:

               precision    recall  f1-score   support

Non-Depressed       0.79      0.78      0.78      5000
    Depressed       0.78      0.79      0.78      5000

     accuracy                           0.78     10000
    macro avg       0.78      0.78      0.78     10000
 weighted avg       0.78      0.78      0.78     10000



['knn_model.pkl']

SVM

In [30]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from scipy.sparse import load_npz, issparse
import pandas as pd
import numpy as np
import joblib

X = load_npz('F:/427_Project/subset_final_feature_matrix.npz')
y = pd.read_csv('F:/427_Project/subset_final_labels.csv')['label'].values

if issparse(X):
    print("NaN values in X:", np.isnan(X.data).any())
    print("Infinite values in X:", np.isinf(X.data).any())
    if np.isnan(X.data).any():
        print("Replacing NaN with 0.")
        X.data = np.nan_to_num(X.data, nan=0.0)
    if np.isinf(X.data).any():
        print("Replacing infinite values with 1e6.")
        X.data = np.where(np.isinf(X.data), 1e6, X.data)
    X = X.astype(np.float32)
else:
    print("NaN values in X:", np.isnan(X).any())
    print("Infinite values in X:", np.isinf(X).any())
    if np.isnan(X).any():
        print("Replacing NaN with 0.")
        X = np.nan_to_num(X, nan=0.0)
    if np.isinf(X).any():
        print("Replacing infinite values with 1e6.")
        X = np.where(np.isinf(X), 1e6, X)
    X = X.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = LinearSVC(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_pred_scores = model.decision_function(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_scores))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


joblib.dump(model, 'F:/427_Project/linear_svc_model.pkl')

NaN values in X: False
Infinite values in X: False
Classification Report:
               precision    recall  f1-score   support

         0.0       0.88      0.91      0.90      5000
         1.0       0.91      0.88      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.90      0.89      0.89     10000
weighted avg       0.90      0.89      0.89     10000

ROC-AUC Score: 0.95770048
Confusion Matrix:
 [[4560  440]
 [ 614 4386]]


['F:/427_Project/linear_svc_model.pkl']

NAIVE BAYES

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, save_npz
import re
from textblob import TextBlob
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from scipy.sparse import load_npz, issparse
import joblib


dataset = pd.read_csv('F:/427_Project/subset_data.csv')

dataset['body'] = dataset['body'].fillna(dataset['title'])
dataset['created_utc'] = dataset['created_utc'].fillna(dataset['created_utc'].median())
dataset['num_comments'] = dataset['num_comments'].fillna(0)
dataset['upvotes'] = dataset['upvotes'].fillna(0)


def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)  
    text = re.sub(r'[^\w\s]', '', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

dataset['cleaned_text'] = dataset['body'].apply(clean_text)


tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_text = tfidf.fit_transform(dataset['cleaned_text'])

dataset['upvotes'] = dataset['upvotes'].clip(lower=0, upper=dataset['upvotes'].quantile(0.95))
dataset['num_comments'] = dataset['num_comments'].clip(lower=0, upper=dataset['num_comments'].quantile(0.95))
dataset['log_upvotes'] = np.log1p(dataset['upvotes'])
dataset['log_num_comments'] = np.log1p(dataset['num_comments'])
dataset['upvotes_per_comment'] = np.where(dataset['num_comments'] == 0, 0, dataset['upvotes'] / dataset['num_comments'])
dataset['upvotes_per_comment'] = dataset['upvotes_per_comment'].clip(upper=dataset['upvotes_per_comment'].quantile(0.95))


dataset['sentiment'] = dataset['cleaned_text'].apply(lambda x: TextBlob(x).sentiment.polarity if x else 0)
dataset['sentiment'] = (dataset['sentiment'] + 1) / 2 

dataset['text_length'] = dataset['cleaned_text'].apply(lambda x: len(x.split()))


numerical_features = dataset[['log_upvotes', 'log_num_comments', 'upvotes_per_comment', 'sentiment', 'text_length']].values
X_numerical = np.nan_to_num(numerical_features, nan=0.0, posinf=1e6, neginf=0.0)  
X = hstack([X_text, X_numerical])


save_npz('F:/427_Project/subset_final_feature_matrix_nb.npz', X)
dataset[['label']].to_csv('F:/427_Project/subset_final_labels_nb.csv', index=False)
print("Preprocessed subset shape:", X.shape)


X = load_npz('F:/427_Project/subset_final_feature_matrix_nb.npz')
y = pd.read_csv('F:/427_Project/subset_final_labels_nb.csv')['label'].values

if issparse(X):
    print("Negative values in X:", (X.data < 0).any())
    print("NaN values in X:", np.isnan(X.data).any())
    print("Infinite values in X:", np.isinf(X.data).any())
    if np.isnan(X.data).any():
        print("Replacing NaN with 0.")
        X.data = np.nan_to_num(X.data, nan=0.0)
    if np.isinf(X.data).any():
        print("Replacing infinite values with 1e6.")
        X.data = np.where(np.isinf(X.data), 1e6, X.data)
    if (X.data < 0).any():
        print("Shifting negative values to 0.")
        X.data = np.maximum(X.data, 0)
    X = X.astype(np.float32)
else:
    print("Negative values in X:", (X < 0).any())
    print("NaN values in X:", np.isnan(X).any())
    print("Infinite values in X:", np.isinf(X).any())
    if np.isnan(X).any():
        print("Replacing NaN with 0.")
        X = np.nan_to_num(X, nan=0.0)
    if np.isinf(X).any():
        print("Replacing infinite values with 1e6.")
        X = np.where(np.isinf(X), 1e6, X)
    if (X < 0).any():
        print("Shifting negative values to 0.")
        X = np.maximum(X, 0)
    X = X.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


model = MultinomialNB()
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

joblib.dump(model, 'F:/427_Project/naive_bayes_model.pkl')

Preprocessed subset shape: (50000, 5005)
Negative values in X: False
NaN values in X: False
Infinite values in X: False
Classification Report:
               precision    recall  f1-score   support

         0.0       0.70      0.89      0.78      5000
         1.0       0.85      0.62      0.71      5000

    accuracy                           0.75     10000
   macro avg       0.77      0.75      0.75     10000
weighted avg       0.77      0.75      0.75     10000

ROC-AUC Score: 0.8356108400000001
Confusion Matrix:
 [[4434  566]
 [1910 3090]]


['F:/427_Project/naive_bayes_model.pkl']